<a href="https://colab.research.google.com/github/vikramaH/links/blob/main/PDFs_In_Python_using_PyMuPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

01-08-2026

### See

https://pypi.org/project/pymupdf/

https://github.com/pymupdf/pymupdf

**PDFs are everywhere** — whether it’s research papers, invoices, contracts, or e-books. Despite their ubiquity, working with PDFs **programmatically** has often been considered a challenge. That’s where PyMuPDF (also known as fitz) comes in.

    ==> 108 million downloads per months

    ===> Its extracted text has less loss of information than that done with I_Love_PDF.

### Great features

There are many Python libraries for handling PDFs (like PyPDF2, pdfplumber, or ReportLab). PyMuPDF stands out because:

    Speed and Efficiency — Built on the lightweight MuPDF library, it offers faster text and image extraction compared to many alternatives.
    
    Feature Rich — Supports not just PDF, but also XPS, EPUB, MOBI, and even image formats.

    Precise Layout Extraction — Provides detailed access to text, fonts, positions, and metadata, making it ideal for data mining or AI/ML applications.

    Graphics and Rendering — Enables you to render pages as images, annotate them, and even draw shapes directly on documents.

    Cross-Platform — Works on Windows, macOS, and Linux.



    Fast — powered by MuPDF, a best-in-class C rendering engine (10 to 50 times faster than pypdf)

    Accurate — pixel-perfect text extraction with font, color, and position metadata

    Versatile — read, write, annotate, redact, merge, split, and convert documents

    LLM-ready — native Markdown output via PyMuPDF4LLM for RAG and AI pipelines

    No mandatory dependencies — pip install pymupdf and you're done

Read PDF Metadata: PyMuPDF supports accessing metadata of PDF documents containing information such as author, title, subject and creation date etc.

Modify PDF Metadata: The library also allows modifying metadata of PDF documents.

Read XML Metadata: PDF documents also contain XML metadata which is not limited to standard document properties like author, title etc. and can have additional metadata. With PyMuPDF, developers can also read it.

Change XML Metadata: Developers can also change XML metadata of PDFs using PyMuPDF library.

# "PDFs in Python"
> "A quick-start guide for working with PyMuPDF"

- toc: true
- branch: master
- badges: true
- comments: true
- author: David Cato
- categories: [jupyter, python, quick-demo]

This notebook primarily intended as a quick reference for working with PDFs in Python, to be expanded over time. The structure and much of the content is based on following this [tutorial](https://pymupdf.readthedocs.io/en/latest/tutorial/) in the PyMuPDF docs.

**PyMuPDF**:
- [GitHub](https://github.com/pymupdf/PyMuPDF)
- [Docs](https://pymupdf.readthedocs.io/en/latest/)
- Recipes:
  - [Docs - Recipes](https://pymupdf.readthedocs.io/en/latest/faq/#faq)
  - [Wiki - Recipes](https://github.com/pymupdf/PyMuPDF/wiki) (e.g. [working with SVGs](https://github.com/pymupdf/PyMuPDF/wiki/Vector-Image-Support), [extract fonts](https://github.com/pymupdf/PyMuPDF/wiki/How-to-Extract-Fonts-from-a-PDF), [extract text from rectangle](https://github.com/pymupdf/PyMuPDF/wiki/How-to-extract-text-from-a-rectangle))
  - [GitHub - Utilities](https://github.com/pymupdf/PyMuPDF-Utilities/tree/master/demo) (e.g. [demo.py](https://github.com/pymupdf/PyMuPDF-Utilities/blob/master/demo/demo.py) - python script similar to this notebook)
- Supported formats:
  - PDF, XPS, OpenXPS, CBZ, CBR, FB2, EPUB

## Installation

[Installing with pip](https://pypi.org/project/PyMuPDF/#files):

In [ ]:
!pip install PyMuPDF

## Import (`fitz`) & Version Info

In [ ]:
import fitz

In [ ]:
print(fitz.__doc__)

## Working with Documents

### Open Document

First, download a document to work with. Note the use of `joblib` to cache the response, which saves us time on reloading the notebook and also is nice by not hitting the server again):

In [ ]:
from joblib import Memory
from pathlib import Path
# !pip install requests
import requests

path = Path('.')
CACHE_DIR =  path / '.jupyter_cache'
memory = Memory(CACHE_DIR, verbose=0)

@memory.cache
def download(url, dst):
    response = requests.get(url, allow_redirects=True)
    with open(dst, 'wb') as f:
        f.write(response.content)

url = 'https://ai2-website.s3.amazonaws.com/publications/Siegel16eccv.pdf'
fn = path / 'example.pdf'

download(url, fn)

fn

In [ ]:
doc = fitz.open(fn)

### Close Document

In [ ]:
# doc.close()

### Read Meta Data

In [ ]:
doc.page_count, doc.metadata, doc.get_toc()

## Working with Pages

### Read Pages

In [ ]:
# index by page numer
page_no = 0
page = doc[page_no]

page

In [ ]:
# iterate over pages
for page in doc:
    pass

# slice over pages
for page in doc.pages(2,6):
    pass

### Inspect a Page

#### links

In [ ]:
# all links in one page
links = page.get_links()

# iterator over links
for link in page.links():
    pass

links

#### annotations & form fields

In [ ]:
# iterate over annotations
for annot in page.annots():
    print(annot)

# iterate over form fields
for field in page.widgets():
    print(field)

### Convert Page to Image

`pix` is a Pixmap object which (in this case) contains an RGB image of the page, ready to be used for many purposes. Method `Page.getPixmap()` offers lots of variations for controlling the image: resolution, colorspace (e.g. to produce a grayscale image or an image with a subtractive color scheme), transparency, rotation, mirroring, shifting, shearing, etc. For example: to create an RGBA image (i.e. containing an alpha channel), specify `pix = page.getPixmap(alpha=True)`.

#### Choose Resolution

In [ ]:
# default (poor resolution causes text in example.pdf to be barely readable)
# file size: 120 kB
pix = page.get_pixmap()

# 2x default resolution (text is clear, image text still hard to read)
# file size: 328 kB
zoom_xy = (2., 2.)
mat = fitz.Matrix(*zoom_xy)
pix = page.get_pixmap(matrix=mat)  # use 'mat' instead of the identity matrix

# 4x default resolution (image text is barely readable)
# file size: 691 kB
zoom_xy = (4., 4.)
mat = fitz.Matrix(*zoom_xy)
pix = page.get_pixmap(matrix=mat)  # use 'mat' instead of the identity matrix

# 8x default resolution (image text is pretty clear but still not perfect)
# file size: 1.4 MB
zoom_xy = (8., 8.)
mat = fitz.Matrix(*zoom_xy)
pix = page.get_pixmap(matrix=mat)  # use 'mat' instead of the identity matrix

#### Save Page as PNG

In [ ]:
dst = fn.parent / f'{fn.stem}_page-{page.number}.png'

dst

In [ ]:
pix.save(str(dst))

#### Open page with Pillow

In [ ]:
from PIL import Image

mode = "RGBA" if pix.alpha else "RGB"
img = Image.frombytes(mode, [pix.width, pix.height], pix.samples)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,20))
plt.imshow(img);

### Extract Text & Images

Use one of the following strings for opt to obtain different formats [2]:

    “text”: (default) plain text with line breaks. No formatting, no text position details, no images.
    “blocks”: generate a list of text blocks (= paragraphs).
    “words”: generate a list of words (strings not containing spaces).
    “html”: creates a full visual version of the page including any images. This can be displayed with your internet browser.
    “dict” / “json”: same information level as HTML, but provided as a Python dictionary or resp. JSON string. See TextPage.extractDICT() resp. TextPage.extractJSON() for details of its structure.
    “rawdict”: a super-set of TextPage.extractDICT(). It additionally provides character detail information like XML. See TextPage.extractRAWDICT() for details of its structure.
    “xhtml”: text information level as the TEXT version but includes images. Can also be displayed by internet browsers.
    “xml”: contains no images, but full position and font information down to each single text character. Use an XML module to interpret.

To get an idea about the output of these alternatives, see [Appendix 2: Details on Text Extraction](https://pymupdf.readthedocs.io/en/latest/app2/#appendix2).

In [ ]:
text_options = {
    'text', 'blocks', 'words', 'html',
    'dict', 'json', 'rawDict', 'xhtml', 'xml'}

opt = 'text'

text = page.get_text(opt)

text

### Search for Text

In [ ]:
rectangles = page.search_for('We')

rectangles

## More Features...

- PDF Maintenance: can only modify in PDF format, first convert to PDF using `doc.convertToPDF()`, after modifying, save to disk with `doc.save()`.
- Join & Split PDF documents
- Modify, Create, Re-arrange & Delete PDF pages
- Embed arbitrary data (similar to ZIP files)